# 04 - Environment Verification Station

Notebook này dùng để kiểm định (Sanity Check) môi trường RL trước khi chạy huấn luyện chính thức. 
Nó sử dụng trực tiếp mã nguồn từ `src/rl/environments/traffic_env.py`.

In [6]:
from pathlib import Path
import torch
import pandas as pd
import numpy as np
import sys

ROOT = Path('/workspace/ai-core')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.rl.environments.traffic_env import TrafficForecastingEnv
from src.ml.data.dataset import prepare_dataloaders
from src.ml.feature_contract import NUM_CLASSES

print("✅ Imports successful.")

✅ Imports successful.


## 1. Load Data & Initialize Environment
Sử dụng dữ liệu đã cân bằng từ Notebook 02 để làm tập mẫu cho môi trường.

In [7]:
df_path = ROOT / 'data' / 'processed' / '02_balanced_training_data.parquet'
df = pd.read_parquet(df_path)

# Tạo một dataloader nhỏ để test
train_loader, val_loader, _, _ = prepare_dataloaders(
    df.head(10000), # Chỉ lấy 10k dòng để test nhanh
    batch_size=32,
    use_weighted_sampler=False
)

# Khởi tạo môi trường thật
env = TrafficForecastingEnv(
    dataloader=val_loader,
    class_weights=[1.0, 1.0, 1.0, 2.0, 5.0, 10.0], # Ưu tiên kẹt xe nặng
    reward_scale=1.0
)

obs, info = env.reset()
print("✅ Environment initialized.")
print("Observation keys:", obs.keys())
print("Dynamic shape:", obs['dynamic'].shape)

Tổng số dòng dữ liệu thô: 10000
Tổng số cửa sổ hợp lệ thu được: 769 (window=12, target_offset=1)
Phân bổ Class trong các cửa sổ:
  - Class 0: 769 windows
  - Class 1: 0 windows
  - Class 2: 0 windows
  - Class 3: 0 windows
  - Class 4: 0 windows
  - Class 5: 0 windows
Dataset split complete (LEAK-PROOF CHRONOLOGICAL): Train=615, Val=154

📊 PHÂN BỔ CỬA SỔ CHI TIẾT (TRAIN VS VAL):
  - Class 0: Train=   615 | Val=   154
  - Class 1: Train=     0 | Val=     0
  - Class 2: Train=     0 | Val=     0
  - Class 3: Train=     0 | Val=     0
  - Class 4: Train=     0 | Val=     0
  - Class 5: Train=     0 | Val=     0
--------------------------------------------------
✅ Environment initialized.
Observation keys: dict_keys(['dynamic', 'static', 'categorical'])
Dynamic shape: (12, 3)


## 2. Reward Sanity Check
Kiểm tra xem hệ thống phần thưởng có phạt đúng các trường hợp "Vỡ trận" (Lớp 5) không.

In [8]:
def test_reward_scenario(action, target):
    reward, breakdown = env._calculate_reward_details(action, target)
    print(f"Scenario: Action={action} | Target={target}")
    print(f"  - Reward: {reward}")
    print(f"  - Breakdown: {breakdown}")
    print("-" * 30)

# TH 1: Dự báo đúng Class 5
test_reward_scenario(action=5, target=5)

# TH 2: Dự báo là đường thông (0) nhưng thực tế vỡ trận (5) -> Phạt nặng nhất!
test_reward_scenario(action=0, target=5)

# TH 3: Dự báo là vỡ trận (5) nhưng thực tế đường thông (0) -> Phạt báo động giả
test_reward_scenario(action=5, target=0)

Scenario: Action=5 | Target=5
  - Reward: 30.0
  - Breakdown: {'match_bonus': 100.0, 'near_miss_penalty': 0.0, 'far_miss_penalty': 0.0, 'severe_mismatch_penalty': 0.0, 'target_weight': 10.0, 'raw_reward': 100.0, 'scaled_reward': 30.0}
------------------------------
Scenario: Action=0 | Target=5
  - Reward: -30.0
  - Breakdown: {'match_bonus': 0.0, 'near_miss_penalty': 0.0, 'far_miss_penalty': -250.0, 'severe_mismatch_penalty': -200.0, 'target_weight': 10.0, 'raw_reward': -450.0, 'scaled_reward': -30.0}
------------------------------
Scenario: Action=5 | Target=0
  - Reward: -30.0
  - Breakdown: {'match_bonus': 0.0, 'near_miss_penalty': 0.0, 'far_miss_penalty': -25.0, 'severe_mismatch_penalty': -5.0, 'target_weight': 1.0, 'raw_reward': -30.0, 'scaled_reward': -30.0}
------------------------------


## 3. Step Test
Chạy thử 1 bước thật sự của môi trường.

In [9]:
action = 3 # Giả sử Agent chọn Class 3
next_obs, reward, terminated, truncated, info = env.step(action)

print("Step Result:")
print(f"  - Actual Label: {info['actual_label']}")
print(f"  - Reward received: {reward}")
print(f"  - Terminated: {terminated}")

Step Result:
  - Actual Label: 0
  - Reward received: -15.0
  - Terminated: False
